# TDE: Sistema de RH com SQLAlchemy

**Contexto:** time de Sistemas de Informação de uma empresa. O objetivo é construir, em três níveis de profundidade crescente, um pequeno sistema de RH sobre um banco SQLite, cobrindo desde SQL puro seguro até o ORM completo do SQLAlchemy.

Conteúdo abordado:
- Nível 1 (Básico): conexão, SQL puro com `text()` e proteção contra SQL injection
- Nível 2 (Intermediário): SQLAlchemy Core (`Table`, `MetaData`, `insert`, `update`, `select`)
- Nível 3 (Avançado): ORM (`declarative_base`, `relationship`, `Session`)

Cada nível grava dados no mesmo arquivo `sistema_rh.db`, gerado nesta mesma pasta ao rodar o notebook.

In [ ]:
import os

if os.path.exists('sistema_rh.db'):
    os.remove('sistema_rh.db')

## Nível 1 (Básico): Configuração e SQL Puro com Segurança

**Passo 1:** conexão com o banco local via `create_engine('sqlite:///sistema_rh.db')`.
**Passo 2:** transação com `with engine.begin() as conn:` e `text()` para `CREATE TABLE funcionarios` em SQL puro (`id`, `nome`, `cargo`, `salario`).

In [ ]:
from sqlalchemy import create_engine, text

engine = create_engine('sqlite:///sistema_rh.db')

with engine.begin() as conn:
    conn.execute(text('''
        CREATE TABLE funcionarios (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nome TEXT NOT NULL,
            cargo TEXT NOT NULL,
            salario REAL NOT NULL
        )
    '''))

print('tabela funcionarios criada')

**Passo 3 (Segurança):** simula a inserção de um funcionário vindo de um formulário web. O comando usa placeholders nomeados (`:nome`, `:cargo`, `:salario`) e um dicionário de valores, nunca concatenação de string.

**Pergunta reflexiva:** por que nunca concatenar strings direto no SQL? Porque um valor digitado pelo usuário, por exemplo `'; DROP TABLE funcionarios; --`, viraria parte do comando SQL executado se fosse colado direto na string. Os placeholders separam o comando (estrutura fixa, definida no código) dos dados (valores variáveis, enviados à parte pelo driver do banco), então o banco nunca interpreta o conteúdo de `nome` ou `cargo` como comando, só como valor literal. Isso elimina a injeção de SQL por construção, não por filtragem de caracteres perigosos.

In [ ]:
novo_funcionario = {'nome': 'Ana Souza', 'cargo': 'Desenvolvedor Júnior', 'salario': 4200.00}

with engine.begin() as conn:
    conn.execute(
        text('INSERT INTO funcionarios (nome, cargo, salario) VALUES (:nome, :cargo, :salario)'),
        novo_funcionario,
    )

funcionarios_iniciais = [
    {'nome': 'Bruno Lima', 'cargo': 'Desenvolvedor Júnior', 'salario': 4300.00},
    {'nome': 'Carla Dias', 'cargo': 'Analista de RH', 'salario': 5100.00},
    {'nome': 'Diego Alves', 'cargo': 'Gerente de TI', 'salario': 9800.00},
    {'nome': 'Elisa Prado', 'cargo': 'Desenvolvedor Júnior', 'salario': 4250.00},
]

with engine.begin() as conn:
    for f in funcionarios_iniciais:
        conn.execute(
            text('INSERT INTO funcionarios (nome, cargo, salario) VALUES (:nome, :cargo, :salario)'),
            f,
        )

print('funcionarios inseridos com sucesso, usando placeholders seguros')

**Passo 4:** validação com `pd.read_sql_query()`, que já retorna a tabela como DataFrame.

In [ ]:
import pandas as pd

df_funcionarios = pd.read_sql_query('SELECT * FROM funcionarios', engine)
df_funcionarios

## Nível 2 (Intermediário): SQLAlchemy Core

**Passo 1:** tabela `projetos` definida de forma programática com `Table`, `MetaData` e `Column`, criada fisicamente com `metadata.create_all(engine)`.

In [ ]:
from sqlalchemy import MetaData, Table, Column, Integer, String, Float, ForeignKey

metadata = MetaData()

# funcionarios foi criada via SQL puro (fora deste MetaData), entao precisa
# ser refletida aqui antes de outra tabela poder referencia-la com ForeignKey
funcionarios_tbl = Table('funcionarios', metadata, autoload_with=engine)

projetos = Table(
    'projetos',
    metadata,
    Column('id', Integer, primary_key=True, autoincrement=True),
    Column('nome_projeto', String, nullable=False),
    Column('funcionario_id', Integer, ForeignKey('funcionarios.id')),
    Column('orcamento', Float, nullable=False),
)

metadata.create_all(engine)

print('tabela projetos criada')

**Passo 2:** inserção em lote (bulk insert) de uma lista de dicionários com `conn.execute(insert(projetos), [...])`.

In [ ]:
from sqlalchemy import insert

lista_de_projetos = [
    {'nome_projeto': 'Portal do Colaborador', 'funcionario_id': 1, 'orcamento': 32000.00},
    {'nome_projeto': 'Migração de Servidores', 'funcionario_id': 4, 'orcamento': 78000.00},
    {'nome_projeto': 'App de Ponto Eletrônico', 'funcionario_id': 2, 'orcamento': 21000.00},
]

with engine.begin() as conn:
    conn.execute(insert(projetos), lista_de_projetos)

print(f'{len(lista_de_projetos)} projetos inseridos em lote')

**Passo 3:** reajuste salarial. `update(tabela).where(...).values(...)` aumenta o salário só de quem ocupa o cargo 'Desenvolvedor Júnior'.

In [ ]:
from sqlalchemy import update

with engine.begin() as conn:
    resultado = conn.execute(
        update(funcionarios_tbl)
        .where(funcionarios_tbl.c.cargo == 'Desenvolvedor Júnior')
        .values(salario=funcionarios_tbl.c.salario * 1.10)
    )

print(f'{resultado.rowcount} funcionarios reajustados')
pd.read_sql_query('SELECT nome, cargo, salario FROM funcionarios', engine)

**Passo 4:** relatório salarial. `select` combinando `func.avg()` e `group_by()` para média salarial por cargo.

In [ ]:
from sqlalchemy import select, func

consulta_relatorio = (
    select(funcionarios_tbl.c.cargo, func.avg(funcionarios_tbl.c.salario).label('salario_medio'))
    .group_by(funcionarios_tbl.c.cargo)
)

with engine.connect() as conn:
    relatorio = conn.execute(consulta_relatorio).all()

pd.DataFrame(relatorio, columns=['cargo', 'salario_medio'])

## Nível 3 (Avançado): ORM

**Passo 1 e 2:** classes Python mapeadas via `declarative_base()`, com `mapped_column` e `relationship` ligando `Departamento` e `FuncionarioORM` por `ForeignKey`.

As classes ORM usam tabelas novas (`departamentos` e `funcionarios_orm`), separadas das tabelas `funcionarios`/`projetos` criadas nos níveis anteriores, porque aquelas já existem fisicamente no banco sem a coluna `departamento_id`, então um mapeamento ORM direto sobre elas causaria erro de coluna inexistente. A lógica de mapeamento (Base, mapped_column, relationship, ForeignKey) é a mesma pedida no enunciado.

In [ ]:
from typing import List, Optional
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship


class Base(DeclarativeBase):
    pass


class Departamento(Base):
    __tablename__ = 'departamentos'

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    nome: Mapped[str] = mapped_column(String(50), unique=True)

    funcionarios: Mapped[List['FuncionarioORM']] = relationship(back_populates='departamento')


class FuncionarioORM(Base):
    __tablename__ = 'funcionarios_orm'

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    nome: Mapped[str] = mapped_column(String(100))
    cargo: Mapped[str] = mapped_column(String(50))
    salario: Mapped[float] = mapped_column(Float)
    departamento_id: Mapped[Optional[int]] = mapped_column(ForeignKey('departamentos.id'))

    departamento: Mapped[Optional['Departamento']] = relationship(back_populates='funcionarios')


Base.metadata.create_all(engine)

print('tabelas departamentos e funcionarios_orm criadas')

**Passo 3:** `sessionmaker(bind=engine)` cria a fábrica de sessões. Um `Departamento` é criado, funcionários são adicionados a ele e tudo é persistido de uma vez com `sessao.add()` e `sessao.commit()`.

In [ ]:
from sqlalchemy.orm import sessionmaker

Session = sessionmaker(bind=engine)
sessao = Session()

ti = Departamento(nome='TI')
ti.funcionarios.append(FuncionarioORM(nome='Fábio Rocha', cargo='Desenvolvedor Pleno', salario=7200.00))
ti.funcionarios.append(FuncionarioORM(nome='Giovana Melo', cargo='Analista de Suporte', salario=5300.00))

rh = Departamento(nome='RH')
rh.funcionarios.append(FuncionarioORM(nome='Hugo Castro', cargo='Analista de RH', salario=5100.00))

sessao.add(ti)
sessao.add(rh)
sessao.commit()

print('departamentos e funcionarios persistidos')

**Passo 4:** consulta orientada a objetos. `sessao.execute(select(...)).scalars().all()` retorna objetos `FuncionarioORM` instanciados, navegando de `Departamento` para `FuncionarioORM` via `relationship`. Ao final, `sessao.close()` encerra a sessão.

In [ ]:
from sqlalchemy import select as select_orm

consulta_ti = (
    select_orm(FuncionarioORM)
    .join(Departamento)
    .where(Departamento.nome == 'TI')
)

funcionarios_ti = sessao.execute(consulta_ti).scalars().all()

for f in funcionarios_ti:
    print(f'{f.nome} - {f.cargo} - departamento: {f.departamento.nome}')

sessao.close()